In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Decision Tree Regressor Imputer (`models/imputer.ipynb`)

This notebook trains a **Decision Tree Regressor Imputer** (`rpart`) to predict and fill missing values (`NA`) in clinical vital sign features:
- **Target Vital Sign Features**: `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`.
- **Model Framework**: Regression Trees (`rpart(..., method = "anova")`).
- **Predictor Features**: Patient `age`, `gender`, and remaining non-missing vital sign indicators.
- **Artifact Export**: Saves the trained decision tree imputer object to `deploy/decision_tree_imputer.rds` for downstream integration into `models/lr_extreme.ipynb` and `models/rf_extreme.ipynb`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(rpart)
library(caret)
library(dplyr)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Dataset & Analyze Missing Vital Sign Frequencies
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)

vital_cols <- c("triage_vital_hr", "triage_vital_sbp", "triage_vital_rr", "triage_vital_o2")
vital_cols <- intersect(vital_cols, names(raw_df))

cat("\nMissing Value Frequencies Across Vital Signs:\n")
for (col in vital_cols) {
  n_na <- sum(is.na(raw_df[[col]]))
  pct_na <- (n_na / nrow(raw_df)) * 100
  cat(sprintf("  - %-20s: %6d missing rows (%.2f%%)\n", col, n_na, pct_na))
}

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Train Decision Tree Regressors for Each Vital Sign
# ---------------------------------------------------------
set.seed(config$training$random_state)

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0

df_features <- data.frame(
  age    = raw_df$age,
  gender = gender_vec
)

for (v_col in vital_cols) {
  df_features[[v_col]] <- raw_df[[v_col]]
}

tree_models <- list()
imputation_metrics <- list()

for (target_v in vital_cols) {
  cat(sprintf("\nTraining Decision Tree Regressor Imputer for '%s'...\n", target_v))
  
  # Non-missing subset for training
  valid_df <- df_features[!is.na(df_features[[target_v]]), ]
  
  # Partition into train (80%) and validation (20%)
  in_tr <- createDataPartition(valid_df[[target_v]], p = 0.8, list = FALSE)
  tr_subset  <- valid_df[in_tr, ]
  val_subset <- valid_df[-in_tr, ]
  
  predictor_cols <- setdiff(names(df_features), target_v)
  formula_tree <- as.formula(paste(target_v, "~", paste(predictor_cols, collapse = " + ")))
  
  # Fit Decision Tree Regressor using rpart (anova method)
  tree_fit <- rpart(
    formula = formula_tree,
    data = tr_subset,
    method = "anova",
    control = rpart.control(cp = 0.001, maxdepth = 10)
  )
  
  # Benchmark on validation subset
  val_preds <- predict(tree_fit, newdata = val_subset)
  val_actuals <- val_subset[[target_v]]
  
  rmse <- sqrt(mean((val_preds - val_actuals)^2, na.rm = TRUE))
  mae  <- mean(abs(val_preds - val_actuals), na.rm = TRUE)
  
  cat(sprintf("  -> Validation RMSE: %.4f | MAE: %.4f\n", rmse, mae))
  
  tree_models[[target_v]] <- tree_fit
  imputation_metrics[[target_v]] <- list(rmse = rmse, mae = mae)
}

cat("\nAll Decision Tree Regressor Imputers successfully trained!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Define Imputation Prediction Function & Validate Execution
# ---------------------------------------------------------
impute_missing_vitals <- function(df, imputer_models) {
  df_imputed <- df
  for (v_col in names(imputer_models)) {
    if (v_col %in% names(df_imputed)) {
      na_idx <- which(is.na(df_imputed[[v_col]]))
      if (length(na_idx) > 0) {
        preds <- predict(imputer_models[[v_col]], newdata = df_imputed[na_idx, , drop = FALSE])
        df_imputed[na_idx, v_col] <- preds
      }
    }
  }
  return(df_imputed)
}

# Validate imputation function on dataset
test_imp_df <- impute_missing_vitals(df_features, tree_models)
cat("Remaining missing values after decision tree imputation:\n")
print(colSums(is.na(test_imp_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Save Decision Tree Imputer Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

imputer_obj <- list(
  models = tree_models,
  vital_cols = vital_cols,
  metrics = imputation_metrics
)
class(imputer_obj) <- "decision_tree_imputer"

imputer_path <- file.path(deploy_dir, "decision_tree_imputer.rds")
saveRDS(imputer_obj, file = imputer_path)
cat("Decision Tree Regressor Imputer saved to:", imputer_path, "\n")